# V4 Filtered Medium LightGBM Submission

This version is designed after the v1 Kaggle OOM.

Main changes:

- Drops high-missing, constant, and high-cardinality columns before pandas conversion.
- Writes filtered Polars features to parquet, releases Polars memory, then reads pandas.
- Uses the validated `medium` preset with fixed 1211 trees.
- Keeps a dry-run path for the 10-row public notebook test.

Local validation for the full v4 medium run:

- AUC: `0.8613`
- Gini: `0.7227`
- Stability: `0.7050`
- Features kept after filtering: `554`

In [1]:
from __future__ import annotations

import gc
from pathlib import Path
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
import polars as pl
from pandas.api.types import is_float_dtype, is_integer_dtype

warnings.filterwarnings("ignore")

KAGGLE_ROOT_CANDIDATES = [
    Path("/kaggle/input/home-credit-credit-risk-model-stability"),
    Path("/kaggle/input/competitions/home-credit-credit-risk-model-stability"),
]


def find_local_data_dir() -> Path:
    candidates = [Path("data"), Path.cwd() / "data"]
    candidates.extend(parent / "data" for parent in Path.cwd().resolve().parents)
    for candidate in candidates:
        if (candidate / "sample_submission.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find competition data.")


def find_kaggle_data_dir() -> Path | None:
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None
    for sample_path in input_root.rglob("sample_submission.csv"):
        data_dir = sample_path.parent
        if (data_dir / "parquet_files" / "train" / "train_base.parquet").exists():
            return data_dir
    return None


KAGGLE_ROOT = next((path for path in KAGGLE_ROOT_CANDIDATES if path.exists()), None)
ROOT = KAGGLE_ROOT if KAGGLE_ROOT is not None else (find_kaggle_data_dir() or find_local_data_dir())
WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs/kaggle_v4")
WORKING.mkdir(parents=True, exist_ok=True)

sample_submission = pd.read_csv(ROOT / "sample_submission.csv")
DRY_RUN = sample_submission.shape[0] == 10
PRESET = "static" if DRY_RUN else "medium"
SAMPLE_ROWS = 50000 if DRY_RUN else 0
N_ESTIMATORS = 71 if DRY_RUN else 1211
MAX_MISSING = 0.70
MAX_CAT_UNIQUE = 200
MAX_RSS_GB = 30.0
MIN_AVAILABLE_GB = 8.0

print({
    "root": str(ROOT),
    "working": str(WORKING),
    "dry_run": DRY_RUN,
    "preset": PRESET,
    "sample_rows": SAMPLE_ROWS,
    "n_estimators": N_ESTIMATORS,
})

{'root': 'D:\\A\\credit-risk-kaggle\\data', 'working': 'outputs\\kaggle_v4', 'dry_run': True, 'preset': 'static', 'sample_rows': 50000, 'n_estimators': 71}


In [2]:
import os
import sys


class MemoryLimitExceeded(RuntimeError):
    pass


def _process_memory_mb() -> float | None:
    try:
        import psutil

        return psutil.Process(os.getpid()).memory_info().rss / 1024**2
    except Exception:
        pass

    if sys.platform.startswith("linux") or sys.platform == "darwin":
        try:
            import resource

            rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
            return rss / 1024 if sys.platform.startswith("linux") else rss / 1024**2
        except Exception:
            return None
    return None


def _available_memory_gb() -> float | None:
    try:
        import psutil

        return psutil.virtual_memory().available / 1024**3
    except Exception:
        return None


def log_memory(label: str) -> None:
    rss = _process_memory_mb()
    available = _available_memory_gb()
    parts = [f"[memory] {label}"]
    if rss is not None:
        parts.append(f"rss={rss:.1f} MB")
    if available is not None:
        parts.append(f"available={available:.1f} GB")
    print(" | ".join(parts))


def check_memory(
    label: str,
    max_rss_gb: float = 30.0,
    min_available_gb: float = 8.0,
) -> None:
    rss_mb = _process_memory_mb()
    available_gb = _available_memory_gb()
    log_memory(label)

    if rss_mb is not None and rss_mb / 1024 > max_rss_gb:
        raise MemoryLimitExceeded(
            f"Memory guard stopped at {label}: process RSS {rss_mb / 1024:.1f} GB "
            f"> max_rss_gb={max_rss_gb:.1f} GB"
        )
    if available_gb is not None and available_gb < min_available_gb:
        raise MemoryLimitExceeded(
            f"Memory guard stopped at {label}: available memory {available_gb:.1f} GB "
            f"< min_available_gb={min_available_gb:.1f} GB"
        )

log_memory('initialized')

[memory] initialized | rss=215.8 MB | available=26.6 GB


In [3]:
from dataclasses import dataclass
from pathlib import Path

import polars as pl


SPECIAL_COLUMNS = {"case_id", "num_group1", "num_group2"}


@dataclass(frozen=True)
class FeaturePreset:
    depth0_groups: tuple[str, ...]
    aggregate_groups: tuple[str, ...]
    lite_large_groups: tuple[str, ...] = ()


PRESETS: dict[str, FeaturePreset] = {
    "static": FeaturePreset(
        depth0_groups=("static_0", "static_cb_0"),
        aggregate_groups=(),
    ),
    "baseline": FeaturePreset(
        depth0_groups=("static_0", "static_cb_0"),
        aggregate_groups=(
            "person_1",
            "person_2",
            "applprev_1",
            "applprev_2",
            "debitcard_1",
            "deposit_1",
            "other_1",
            "tax_registry_a_1",
            "tax_registry_b_1",
            "tax_registry_c_1",
            "credit_bureau_b_1",
            "credit_bureau_b_2",
        ),
    ),
    "medium": FeaturePreset(
        depth0_groups=("static_0", "static_cb_0"),
        aggregate_groups=(
            "person_1",
            "person_2",
            "applprev_1",
            "applprev_2",
            "debitcard_1",
            "deposit_1",
            "other_1",
            "tax_registry_a_1",
            "tax_registry_b_1",
            "tax_registry_c_1",
            "credit_bureau_a_1",
            "credit_bureau_b_1",
            "credit_bureau_b_2",
        ),
    ),
    "a2lite": FeaturePreset(
        depth0_groups=("static_0", "static_cb_0"),
        aggregate_groups=(
            "person_1",
            "person_2",
            "applprev_1",
            "applprev_2",
            "debitcard_1",
            "deposit_1",
            "other_1",
            "tax_registry_a_1",
            "tax_registry_b_1",
            "tax_registry_c_1",
            "credit_bureau_a_1",
            "credit_bureau_b_1",
            "credit_bureau_b_2",
        ),
        lite_large_groups=("credit_bureau_a_2",),
    ),
    "a2core": FeaturePreset(
        depth0_groups=("static_0", "static_cb_0"),
        aggregate_groups=(
            "person_1",
            "applprev_1",
            "tax_registry_a_1",
            "credit_bureau_a_1",
        ),
        lite_large_groups=("credit_bureau_a_2",),
    ),
    "a2ultra": FeaturePreset(
        depth0_groups=("static_0", "static_cb_0"),
        aggregate_groups=(
            "person_1",
            "applprev_1",
            "tax_registry_a_1",
        ),
        lite_large_groups=("credit_bureau_a_1", "credit_bureau_a_2"),
    ),
    "full": FeaturePreset(
        depth0_groups=("static_0", "static_cb_0"),
        aggregate_groups=(
            "person_1",
            "person_2",
            "applprev_1",
            "applprev_2",
            "debitcard_1",
            "deposit_1",
            "other_1",
            "tax_registry_a_1",
            "tax_registry_b_1",
            "tax_registry_c_1",
            "credit_bureau_a_1",
            "credit_bureau_a_2",
            "credit_bureau_b_1",
            "credit_bureau_b_2",
        ),
    ),
}


LITE_LARGE_NUMERIC_COLUMNS: dict[str, tuple[str, ...]] = {
    "credit_bureau_a_1": (
        "dpdmax_139P",
        "dpdmax_757P",
        "dpdmaxdateyear_596T",
        "overdueamountmax_155A",
        "overdueamountmax_35A",
        "numberofoverdueinstlmax_1151L",
        "numberofoverdueinstlmax_1039L",
        "numberofinstls_320L",
        "numberofinstls_229L",
        "residualamount_856A",
        "residualamount_488A",
        "totaloutstanddebtvalue_39A",
        "totaloutstanddebtvalue_668A",
        "debtoutstand_525A",
        "debtoverdue_47A",
    ),
    "credit_bureau_a_2": (
        "pmts_dpd_1073P",
        "pmts_dpd_303P",
        "pmts_overdue_1140A",
        "pmts_overdue_1152A",
        "pmts_month_158T",
        "pmts_month_706T",
        "pmts_year_1139T",
        "pmts_year_507T",
    ),
}


def split_dir(data_dir: Path, split: str) -> Path:
    return data_dir / "parquet_files" / split


def files_for_group(data_dir: Path, split: str, group: str) -> list[Path]:
    paths = sorted(split_dir(data_dir, split).glob(f"{split}_{group}*.parquet"))
    if not paths:
        raise FileNotFoundError(f"No parquet files found for split={split!r}, group={group!r}")
    return paths


def scan_group(data_dir: Path, split: str, group: str) -> pl.LazyFrame:
    paths = files_for_group(data_dir, split, group)
    frames = [pl.scan_parquet(str(path)) for path in paths]
    schemas = [frame.collect_schema() for frame in frames]
    target_schema: dict[str, pl.DataType] = {}

    for schema in schemas:
        for col, dtype in schema.items():
            if col not in target_schema or target_schema[col] == pl.Null:
                target_schema[col] = dtype
    for key in ("case_id", "num_group1", "num_group2"):
        if key in target_schema:
            target_schema[key] = pl.Int64
    for col, dtype in list(target_schema.items()):
        if dtype == pl.Null:
            target_schema[col] = pl.Utf8

    normalized = []
    for frame, schema in zip(frames, schemas):
        exprs = []
        for col, target_dtype in target_schema.items():
            if col in schema and schema[col] != target_dtype:
                exprs.append(pl.col(col).cast(target_dtype, strict=False).alias(col))
        normalized.append(frame.with_columns(exprs) if exprs else frame)
    return pl.concat(normalized, how="vertical_relaxed")


def normalize_dates(lf: pl.LazyFrame) -> pl.LazyFrame:
    schema = lf.collect_schema()
    exprs = []
    for name, dtype in schema.items():
        if name.endswith("D") or name == "date_decision":
            if dtype.is_numeric():
                continue
            exprs.append(
                pl.col(name)
                .cast(pl.Utf8)
                .str.strptime(pl.Date, strict=False)
                .cast(pl.Int32)
                .alias(name)
            )
    return lf.with_columns(exprs) if exprs else lf


def load_base(data_dir: Path, split: str) -> pl.LazyFrame:
    return normalize_dates(scan_group(data_dir, split, "base"))


def filter_cases(lf: pl.LazyFrame, case_ids: pl.LazyFrame | None) -> pl.LazyFrame:
    if case_ids is None:
        return lf
    return lf.join(case_ids, on="case_id", how="semi")


def load_depth0(
    data_dir: Path,
    split: str,
    group: str,
    case_ids: pl.LazyFrame | None = None,
) -> pl.LazyFrame:
    lf = normalize_dates(scan_group(data_dir, split, group))
    lf = filter_cases(lf, case_ids)
    return lf.unique(subset=["case_id"], keep="last")


def aggregate_group(
    data_dir: Path,
    split: str,
    group: str,
    case_ids: pl.LazyFrame | None = None,
) -> pl.LazyFrame:
    lf = normalize_dates(scan_group(data_dir, split, group))
    lf = filter_cases(lf, case_ids)
    schema = lf.collect_schema()
    aggs: list[pl.Expr] = [pl.len().alias(f"{group}__row_count")]

    for col, dtype in schema.items():
        if col in SPECIAL_COLUMNS:
            continue
        out = f"{group}__{col}"
        if dtype.is_numeric() or dtype == pl.Boolean or col.endswith("D"):
            base = pl.col(col).cast(pl.Float64, strict=False)
            aggs.extend(
                [
                    base.mean().alias(f"{out}__mean"),
                    base.max().alias(f"{out}__max"),
                    base.min().alias(f"{out}__min"),
                    base.std().alias(f"{out}__std"),
                ]
            )
        else:
            aggs.append(pl.col(col).n_unique().alias(f"{out}__nunique"))

    return lf.group_by("case_id").agg(aggs)


def aggregate_large_group_lite(
    data_dir: Path,
    split: str,
    group: str,
    case_ids: pl.LazyFrame | None = None,
) -> pl.LazyFrame:
    """Aggregate very large groups file-by-file using only cheap composable stats.

    This avoids the high memory peak of grouping all credit_bureau_a_2 rows at once.
    """
    selected = LITE_LARGE_NUMERIC_COLUMNS[group]
    partials: list[pl.LazyFrame] = []
    for path in files_for_group(data_dir, split, group):
        lf = pl.scan_parquet(str(path))
        schema = lf.collect_schema()
        available = [col for col in selected if col in schema]
        lf = lf.select(["case_id", *available]).with_columns(pl.col("case_id").cast(pl.Int64))
        lf = filter_cases(lf, case_ids)
        aggs: list[pl.Expr] = [pl.len().alias(f"{group}__row_count")]
        for col in available:
            base = pl.col(col).cast(pl.Float64, strict=False)
            prefix = f"{group}__{col}"
            aggs.extend(
                [
                    base.sum().alias(f"{prefix}__sum"),
                    base.count().alias(f"{prefix}__count"),
                    base.max().alias(f"{prefix}__max"),
                    base.min().alias(f"{prefix}__min"),
                ]
            )
        partials.append(lf.group_by("case_id").agg(aggs))

    partial = pl.concat(partials, how="vertical_relaxed")
    schema = partial.collect_schema()
    final_aggs: list[pl.Expr] = [pl.col(f"{group}__row_count").sum().alias(f"{group}__row_count")]
    for col in selected:
        prefix = f"{group}__{col}"
        sum_col = f"{prefix}__sum"
        count_col = f"{prefix}__count"
        max_col = f"{prefix}__max"
        min_col = f"{prefix}__min"
        if sum_col not in schema:
            continue
        final_aggs.extend(
            [
                (pl.col(sum_col).sum() / pl.col(count_col).sum()).alias(f"{prefix}__mean"),
                pl.col(max_col).max().alias(f"{prefix}__max"),
                pl.col(min_col).min().alias(f"{prefix}__min"),
            ]
        )
    return partial.group_by("case_id").agg(final_aggs)


def aggregate_large_group_lite_eager(
    data_dir: Path,
    split: str,
    group: str,
    temp_dir: Path,
    case_ids: pl.DataFrame | None = None,
) -> pl.DataFrame:
    selected = LITE_LARGE_NUMERIC_COLUMNS[group]
    temp_dir.mkdir(parents=True, exist_ok=True)
    partial_paths: list[Path] = []
    case_ids_lf = case_ids.lazy() if case_ids is not None else None

    for idx, path in enumerate(files_for_group(data_dir, split, group)):
        schema = pl.scan_parquet(str(path)).collect_schema()
        available = [col for col in selected if col in schema]
        lf = pl.scan_parquet(str(path)).select(["case_id", *available])
        lf = lf.with_columns(pl.col("case_id").cast(pl.Int64))
        lf = filter_cases(lf, case_ids_lf)
        aggs: list[pl.Expr] = [pl.len().alias(f"{group}__row_count")]
        for col in available:
            base = pl.col(col).cast(pl.Float64, strict=False)
            prefix = f"{group}__{col}"
            aggs.extend(
                [
                    base.sum().alias(f"{prefix}__sum"),
                    base.count().alias(f"{prefix}__count"),
                    base.max().alias(f"{prefix}__max"),
                    base.min().alias(f"{prefix}__min"),
                ]
            )
        partial = lf.group_by("case_id").agg(aggs).collect(engine="streaming")
        out_path = temp_dir / f"{split}_{group}_partial_{idx}.parquet"
        partial.write_parquet(out_path)
        partial_paths.append(out_path)
        del partial

    partial_lf = pl.scan_parquet([str(path) for path in partial_paths])
    schema = partial_lf.collect_schema()
    final_aggs: list[pl.Expr] = [pl.col(f"{group}__row_count").sum().alias(f"{group}__row_count")]
    for col in selected:
        prefix = f"{group}__{col}"
        sum_col = f"{prefix}__sum"
        count_col = f"{prefix}__count"
        max_col = f"{prefix}__max"
        min_col = f"{prefix}__min"
        if sum_col not in schema:
            continue
        final_aggs.extend(
            [
                (pl.col(sum_col).sum() / pl.col(count_col).sum()).alias(f"{prefix}__mean"),
                pl.col(max_col).max().alias(f"{prefix}__max"),
                pl.col(min_col).min().alias(f"{prefix}__min"),
            ]
        )
    return partial_lf.group_by("case_id").agg(final_aggs).collect(engine="streaming")


def build_features(
    data_dir: Path,
    split: str,
    preset_name: str,
    cache_dir: Path | None = None,
    use_cache: bool = True,
    sample_rows: int = 0,
) -> pl.DataFrame:
    if preset_name not in PRESETS:
        raise ValueError(f"Unknown preset {preset_name!r}. Available: {sorted(PRESETS)}")
    preset = PRESETS[preset_name]

    cache_path = None
    if cache_dir is not None and not sample_rows:
        cache_path = cache_dir / f"{split}_{preset_name}_features.parquet"
        if use_cache and cache_path.exists():
            return pl.read_parquet(cache_path)

    lf = load_base(data_dir, split)
    if sample_rows:
        lf = lf.sort("case_id").head(sample_rows)
    case_ids = lf.select("case_id")
    for group in preset.depth0_groups:
        lf = lf.join(load_depth0(data_dir, split, group, case_ids), on="case_id", how="left")
    for group in preset.aggregate_groups:
        lf = lf.join(aggregate_group(data_dir, split, group, case_ids), on="case_id", how="left")
    df = lf.collect(engine="streaming")
    if preset.lite_large_groups:
        case_ids_df = df.select("case_id")
        temp_root = (cache_path.parent if cache_path is not None else Path("outputs/features")) / "_tmp_lite"
        for group in preset.lite_large_groups:
            lite = aggregate_large_group_lite_eager(
                data_dir,
                split,
                group,
                temp_root / f"{split}_{group}",
                case_ids=case_ids_df if sample_rows else None,
            )
            df = df.join(lite, on="case_id", how="left")
    if cache_path is not None and not sample_rows:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        df.write_parquet(cache_path)
    return df

In [4]:
from dataclasses import dataclass

import numpy as np
import pandas as pd
from pandas.api.types import is_float_dtype, is_integer_dtype


@dataclass
class PreprocessState:
    feature_cols: list[str]
    category_maps: dict[str, list[object]]
    fill_values: dict[str, float]
    dropped_columns: dict[str, list[str]]


DROP_ALWAYS = {"case_id", "target"}


def reduce_mem_usage(df: pd.DataFrame, use_float16: bool = False) -> pd.DataFrame:
    for col in df.columns:
        if pd.api.types.is_bool_dtype(df[col]):
            df[col] = df[col].astype("int8")
        elif is_float_dtype(df[col]):
            df[col] = df[col].astype("float16" if use_float16 else "float32")
        elif is_integer_dtype(df[col]) and col != "case_id":
            df[col] = pd.to_numeric(df[col], downcast="integer")
    return df


def fit_preprocessor(
    df: pd.DataFrame,
    *,
    target_col: str = "target",
    max_missing: float = 0.92,
    max_cat_unique: int = 200,
    min_unique: int = 2,
    keep_cols: set[str] | None = None,
    use_float16: bool = False,
) -> PreprocessState:
    keep_cols = keep_cols or {"WEEK_NUM"}
    candidate_cols = [col for col in df.columns if col not in DROP_ALWAYS]
    dropped: dict[str, list[str]] = {
        "missing": [],
        "constant": [],
        "high_cardinality": [],
    }
    selected: list[str] = []

    for col in candidate_cols:
        if col in keep_cols:
            selected.append(col)
            continue
        missing = float(df[col].isna().mean())
        if missing > max_missing:
            dropped["missing"].append(col)
            continue
        nunique = int(df[col].nunique(dropna=True))
        if nunique < min_unique:
            dropped["constant"].append(col)
            continue
        if (pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col])) and nunique > max_cat_unique:
            dropped["high_cardinality"].append(col)
            continue
        selected.append(col)

    category_maps: dict[str, list[object]] = {}
    fill_values: dict[str, float] = {}

    for col in selected:
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            cat = df[col].astype("category")
            category_maps[col] = list(cat.cat.categories)
            df[col] = cat.cat.codes.astype("int32")
        elif is_float_dtype(df[col]) or is_integer_dtype(df[col]):
            fill_values[col] = float(df[col].median()) if df[col].notna().any() else 0.0

    reduce_mem_usage(df, use_float16=use_float16)
    return PreprocessState(
        feature_cols=selected,
        category_maps=category_maps,
        fill_values=fill_values,
        dropped_columns=dropped,
    )


def apply_preprocessor(df: pd.DataFrame, state: PreprocessState, use_float16: bool = False) -> pd.DataFrame:
    missing = [col for col in state.feature_cols if col not in df.columns]
    if missing:
        df = pd.concat([df, pd.DataFrame(np.nan, index=df.index, columns=missing)], axis=1)
    df = df[state.feature_cols].copy()

    for col, categories in state.category_maps.items():
        if col in df.columns:
            mapper = {value: idx for idx, value in enumerate(categories)}
            df[col] = df[col].map(mapper).fillna(-1).astype("int32")

    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].astype("category").cat.codes.astype("int32")
        elif col in state.fill_values:
            df[col] = df[col].fillna(state.fill_values[col])

    reduce_mem_usage(df, use_float16=use_float16)
    return df


def summarize_drops(state: PreprocessState) -> dict[str, int]:
    return {key: len(value) for key, value in state.dropped_columns.items()} | {"kept": len(state.feature_cols)}

In [5]:
def filter_polars_columns(
    df: pl.DataFrame,
    *,
    max_missing: float,
    max_cat_unique: int,
    min_unique: int = 2,
) -> tuple[pl.DataFrame, list[str], dict[str, list[str]]]:
    keep = {"case_id", "target", "WEEK_NUM"}
    dropped = {"missing": [], "constant": [], "high_cardinality": []}
    selected = []
    n_rows = len(df)
    for col, dtype in zip(df.columns, df.dtypes):
        if col in keep:
            selected.append(col)
            continue
        missing = df[col].null_count() / n_rows
        if missing > max_missing:
            dropped["missing"].append(col)
            continue
        nunique = df[col].n_unique()
        if nunique < min_unique:
            dropped["constant"].append(col)
            continue
        if dtype in (pl.String, pl.Categorical) and nunique > max_cat_unique:
            dropped["high_cardinality"].append(col)
            continue
        selected.append(col)
    return df.select(selected), selected, dropped


def select_polars_columns(df: pl.DataFrame, selected: list[str]) -> pl.DataFrame:
    available = [col for col in selected if col in df.columns]
    return df.select(available)


def lgbm_params(n_estimators: int, seed: int = 42) -> dict:
    return {
        "objective": "binary",
        "n_estimators": int(n_estimators),
        "learning_rate": 0.03,
        "num_leaves": 96,
        "max_depth": -1,
        "min_child_samples": 80,
        "subsample": 0.85,
        "subsample_freq": 1,
        "colsample_bytree": 0.75,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "random_state": seed,
        "n_jobs": -1,
        "device_type": "cpu",
        "verbosity": -1,
    }

In [6]:
check_memory("before train build", MAX_RSS_GB, MIN_AVAILABLE_GB)
train_pl = build_features(
    ROOT,
    "train",
    PRESET,
    cache_dir=WORKING / "features",
    use_cache=True,
    sample_rows=SAMPLE_ROWS,
)
print("raw train shape:", train_pl.shape)
check_memory("after train build", MAX_RSS_GB, MIN_AVAILABLE_GB)

train_pl, selected_polars_cols, polars_drops = filter_polars_columns(
    train_pl,
    max_missing=MAX_MISSING,
    max_cat_unique=MAX_CAT_UNIQUE,
)
print("filtered train shape:", train_pl.shape)
print("polars drops:", {key: len(value) for key, value in polars_drops.items()})
check_memory("after polars filter", MAX_RSS_GB, MIN_AVAILABLE_GB)

train_path = WORKING / "train_filtered.parquet"
train_pl.write_parquet(train_path)
del train_pl
gc.collect()
check_memory("after write train parquet", MAX_RSS_GB, MIN_AVAILABLE_GB)

train_pdf = pd.read_parquet(train_path)
check_memory("after read train pandas", MAX_RSS_GB, MIN_AVAILABLE_GB)

y = train_pdf["target"].astype("int8")
state = fit_preprocessor(
    train_pdf,
    max_missing=MAX_MISSING,
    max_cat_unique=MAX_CAT_UNIQUE,
)
X = apply_preprocessor(train_pdf, state)
del train_pdf
gc.collect()
print("X shape:", X.shape)
print("drop summary:", summarize_drops(state))
check_memory("after preprocess", MAX_RSS_GB, MIN_AVAILABLE_GB)

[memory] before train build | rss=217.7 MB | available=26.6 GB
raw train shape: (50000, 224)
[memory] after train build | rss=982.5 MB | available=25.9 GB
filtered train shape: (50000, 107)
polars drops: {'missing': 114, 'constant': 3, 'high_cardinality': 0}
[memory] after polars filter | rss=983.9 MB | available=25.9 GB
[memory] after write train parquet | rss=1010.5 MB | available=25.9 GB
[memory] after read train pandas | rss=1123.3 MB | available=25.8 GB
X shape: (50000, 96)
drop summary: {'missing': 0, 'constant': 9, 'high_cardinality': 0, 'kept': 96}
[memory] after preprocess | rss=1147.3 MB | available=25.7 GB


In [7]:
model = lgb.LGBMClassifier(**lgbm_params(N_ESTIMATORS))
model.fit(X, y)

del X, y
gc.collect()
check_memory("after train model and release train matrix", MAX_RSS_GB, MIN_AVAILABLE_GB)

[memory] after train model and release train matrix | rss=1133.5 MB | available=25.8 GB


In [8]:
test_pl = build_features(
    ROOT,
    "test",
    PRESET,
    cache_dir=WORKING / "features",
    use_cache=True,
)
print("raw test shape:", test_pl.shape)
test_pl = select_polars_columns(test_pl, [col for col in selected_polars_cols if col != "target"])
print("filtered test shape:", test_pl.shape)
check_memory("after test polars filter", MAX_RSS_GB, MIN_AVAILABLE_GB)

test_path = WORKING / "test_filtered.parquet"
test_pl.write_parquet(test_path)
del test_pl
gc.collect()
check_memory("after write test parquet", MAX_RSS_GB, MIN_AVAILABLE_GB)

test_pdf = pd.read_parquet(test_path)
test_ids = test_pdf["case_id"].to_numpy()
X_test = apply_preprocessor(test_pdf, state)
del test_pdf
gc.collect()
check_memory("after test preprocess", MAX_RSS_GB, MIN_AVAILABLE_GB)

pred = model.predict_proba(X_test)[:, 1].astype(np.float32)
submission = pd.DataFrame({"case_id": test_ids, "score": pred})
submission = sample_submission[["case_id"]].merge(submission, on="case_id", how="left")
submission["score"] = submission["score"].fillna(float(np.mean(pred)))
submission.to_csv(WORKING / "submission.csv", index=False)
submission.to_csv("submission.csv", index=False)
check_memory("submission done", MAX_RSS_GB, MIN_AVAILABLE_GB)
print(submission.shape)
submission.head()

raw test shape: (10, 223)
filtered test shape: (10, 106)
[memory] after test polars filter | rss=1085.7 MB | available=25.8 GB
[memory] after write test parquet | rss=1074.6 MB | available=25.8 GB
[memory] after test preprocess | rss=1007.4 MB | available=25.8 GB
[memory] submission done | rss=1007.7 MB | available=25.8 GB
(10, 2)


,case_id,score
0,57543,0.040734
1,57549,0.098715
2,57551,0.031792
3,57552,0.050236
4,57569,0.100481
